In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import matplotlib.animation as animation

from sklearn.preprocessing import MinMaxScaler


import seaborn as sns

import numpy as np

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense,  Dropout
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [ ]:
df = pd.read_csv('data.csv', low_memory=False)

In [ ]:
df

In [ ]:
df = df.dropna()


In [ ]:
df

In [ ]:
df.info()

In [ ]:
df = df.sort_values(by=['MMSI', '# Timestamp'], ascending=[True, True])


In [ ]:
df

In [ ]:
df.describe()


In [ ]:
# Correlation Matrix
print("\nCorrelation Matrix:")
print(df.corr(numeric_only=True))
sns.heatmap(df.corr(numeric_only=True), annot=False, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:

# Grouped Aggregates
print("\nAverage SOG by Ship Type:")
avg_sog = df.groupby('Ship type')['SOG'].mean()
print(avg_sog)
avg_sog.plot(kind='bar', title='Average SOG by Ship Type')
plt.ylabel('Average SOG')
plt.xlabel('Ship Type')
plt.grid(True)
plt.tight_layout()
plt.show()


print("\nAverage Draught by Ship Type:")
avg_draught = df.groupby('Ship type')['Draught'].mean()
print(avg_draught)
avg_draught.plot(kind='bar', title='Average Draught by Ship Type', color='purple')
plt.ylabel('Average Draught')
plt.xlabel('Ship Type')
plt.grid(True)
plt.tight_layout()
plt.show()

# Visualizations per Ship Type
print("\nVisualizations per Ship Type:")
ship_types = df['Ship type'].unique()
for ship in ship_types:
    subset = df[df['Ship type'] == ship]
    plt.figure(figsize=(10, 4))
    plt.hist(subset['SOG'], bins=20, alpha=0.7)
    plt.title(f'SOG Distribution for Ship Type: {ship}')
    plt.xlabel('SOG')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.scatter(subset['Length'], subset['Width'], alpha=0.6)
    plt.title(f'Ship Dimensions for Ship Type: {ship}')
    plt.xlabel('Length')
    plt.ylabel('Width')
    plt.grid(True)
    plt.tight_layout()
    plt.show()


print("\nMax ROT by Navigational Status:")
max_rot = df.groupby('Navigational status')['ROT'].max()
print(max_rot)
max_rot.plot(kind='bar', title='Max ROT by Navigational Status', color='orange')
plt.ylabel('Max ROT')
plt.xlabel('Navigational Status')
plt.grid(True)
plt.tight_layout()
plt.show()

print("\nShip Counts by Cargo Type:")
cargo_counts = df.groupby('Cargo type')['MMSI'].count()
print(cargo_counts)
cargo_counts.plot(kind='bar', title='Ship Counts by Cargo Type', color='green')
plt.ylabel('Count')
plt.xlabel('Cargo Type')
plt.grid(True)
plt.tight_layout()
plt.show()



print("\nAverage Width by Cargo Type:")
avg_width = df.groupby('Cargo type')['Width'].mean()
print(avg_width)
avg_width.plot(kind='bar', title='Average Width by Cargo Type', color='red')
plt.ylabel('Average Width')
plt.xlabel('Cargo Type')
plt.grid(True)
plt.tight_layout()
plt.show()

print("\nAverage Length by Cargo Type:")
avg_length = df.groupby('Cargo type')['Length'].mean()
print(avg_length)
avg_length.plot(kind='bar', title='Average Length by Cargo Type', color='cyan')
plt.ylabel('Average Length')
plt.xlabel('Cargo Type')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# 1. Convert to datetime
df['# Timestamp'] = pd.to_datetime(df['# Timestamp'], dayfirst=True)

# 2. Filter a vessel
most_common_mmsi = df['MMSI'].value_counts().idxmax()

vessel_df = df[df['MMSI'] == most_common_mmsi].copy()

# 3. Set Timestamp as index and sort
vessel_df = vessel_df.sort_values('# Timestamp')
vessel_df.set_index('# Timestamp', inplace=True)

# 4. Select only numeric columns to resample
numeric_vessel_df = vessel_df.select_dtypes(include=['number'])

# 5. Resample to 1-minute intervals and interpolate
resampled_numeric = numeric_vessel_df.resample('1T').mean()
resampled_numeric = resampled_numeric.interpolate(method='linear')

# 6. Add MMSI back if dropped
resampled_numeric['MMSI'] = most_common_mmsi

# 7. Reset index for easier handling
resampled_df = resampled_numeric.reset_index()

# Done!
print(resampled_df[['# Timestamp', 'Latitude', 'Longitude']])

In [ ]:
resampled_df = resampled_df.sort_values("# Timestamp")

# Plot Latitude vs Longitude
plt.figure(figsize=(10, 6))
plt.plot(resampled_df['Longitude'], resampled_df['Latitude'], marker='o', markersize=3, linestyle='-', color='blue')
plt.title(f"Ship Route (MMSI: {resampled_df['MMSI'].iloc[0]})")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True)
plt.show()

In [ ]:
features = ['Latitude', 'Longitude']  # input and target
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(resampled_df[features])

In [ ]:
def create_latlon_sequences(data, window_size=5):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

X, y = create_latlon_sequences(scaled_data, window_size=10)

split_ratio = 0.8
split_index = int(len(X) * split_ratio)

# Split into training and testing sets
X_train = X[:split_index]
y_train = y[:split_index]
X_test = X[split_index:]
y_test = y[split_index:]

In [ ]:
# Improved LSTM Model
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(X.shape[1], X.shape[2])),
    Dropout(0.2),
    LSTM(64),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(2)  # [Latitude, Longitude]
])

model.compile(optimizer='adam', loss='mse')
model.summary()
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=50, batch_size=32)

# Predictions and inverse scaling
y_pred = model.predict(X_test)
y_test_inv = scaler.inverse_transform(y_test)
y_pred_inv = scaler.inverse_transform(y_pred)

# Evaluation Metrics
mse = mean_squared_error(y_test_inv, y_pred_inv)
mae = mean_absolute_error(y_test_inv, y_pred_inv)
r2 = r2_score(y_test_inv, y_pred_inv)

print("\nEvaluation Metrics:")
print("MSE:", mse)
print("MAE:", mae)
print("R^2 Score:", r2)

